In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Introduction to `RSet`

In theoretical computer science, the concept of a *Set* is fundamental to formal models such as automata, grammars, constraint satisfaction problems (CSPs like Sudoku), and propositional logic. Before we can implement algorithms for these structures, we must establish a computationally sound representation of mathematical sets.

## Mathematical Definition
A set is a collection of distinct objects where the order of elements is irrelevant. Formally, for two sets $A$ and $B$, the following holds true:

$$ A = \{1, 2, 3\} $$
$$ B = \{3, 2, 1\} $$
$$ A = B $$

Furthermore, adding an element that already exists in the set does not alter the set. If we define $C = A \cup \{2\}$, then $C$ evaluates to $\{1, 2, 3\}$.

In the following we will examine how these mathematical properties translate to data structures in OCaml.

## Lists Are Not Sets

In OCaml, the operator `=` compares two values *structurally*: Two lists are equal if they contain the same elements in the same order.

In [ ]:
let la = [1; 2; 3]
let lb = [1; 2; 3];;
la = lb

However, lists are not sets: The order of the elements matters and an element can occur more than once.

In [ ]:
[1; 2; 3] = [3; 2; 1]

In [ ]:
[1; 2; 3] = [1; 2; 2; 3]

## The Module `Set` of the Standard Library

The standard library of OCaml provides sets via the *functor* `Set.Make`.  We have already used the functor `Map.Make` in the introduction to OCaml.  The argument of `Set.Make` is a module that defines the type of the elements together with a function `compare` that compares elements.  The module `Int` is such a module.  The toplevel answers with the signature of the new module `IntSet`.

In [ ]:
module IntSet = Set.Make (Int)

The type `IntSet.t` is *abstract*.  Therefore, the toplevel only shows `<abstr>`.  The function `IntSet.elements` returns the elements of a set as a sorted list.  The set `primitive_set` is built by adding the numbers 1, 2, 3, and 2 to the empty set.  As sets in OCaml are *immutable*, `IntSet.add` does not change a set but returns a new set.  The *pipe operator* `|>` passes the set from one call of `IntSet.add` to the next.

In [ ]:
let primitive_set = IntSet.empty |> IntSet.add 1 |> IntSet.add 2 |> IntSet.add 3 |> IntSet.add 2;;
Printf.printf "Primitive set contents: [%s]\n" (String.concat "; " (List.map string_of_int (IntSet.elements primitive_set)));
Printf.printf "Size of the set: %d\n" (IntSet.cardinal primitive_set)

### The Limitation of `Set.Make`

`Set.Make` correctly handles numbers and strings.  However, theoretical computer science frequently requires more complex structures like *sets that contain other sets* (e.g., powerset constructions or partitions).  `Set.Make` has two drawbacks here:

1. For every type of elements, we have to create a new module.  For sets of sets of integers, we need a module `IntSetSet = Set.Make (IntSet)`, for sets of literals a module `LiteralSet`, for sets of sets of literals yet another module, and so on.
2. The operator `=` does not work for these sets.

Let us examine the second point.  We create the set $\{1, 2, 3\}$ in two different ways:

In [ ]:
let set_a = IntSet.of_list [1; 2; 3]
let set_b = IntSet.add 1 (IntSet.of_list [2; 3])

The function `IntSet.equal` confirms that these sets are equal:

In [ ]:
IntSet.equal set_a set_b

However, the operator `=` claims that they are different:

In [ ]:
set_a = set_b

### Analysis of the Output

`Set.Make` stores a set as a *balanced binary search tree*.  The shape of this tree depends on the order in which the elements have been inserted.  The operator `=` compares the trees and not the sets they represent.  Since `set_a` and `set_b` are represented by trees of different shapes, `=` considers them to be different.  The same holds for the polymorphic function `compare`.

This is a critical failure if we want to work with sets of sets.  For example, the function `List.sort_uniq compare` sorts a list and removes duplicates.  Mathematically, $\{ \{1, 2, 3\}, \{1, 2, 3\} \}$ must simplify to $\{ \{1, 2, 3\} \}$.  But the list `[set_a; set_b]` still has two elements after the duplicates have been removed:

In [ ]:
List.length (List.sort_uniq compare [set_a; set_b])

## Introducing Value Semantics

We require a data structure that guarantees **value semantics**: Two sets have to be considered equal if their mathematical contents are identical, regardless of how they have been constructed.

For this purpose, we use the module `RSet`, which is defined in the file `rset.ml` in the parent directory.  The file `setup.ml`, which is loaded in the first cell of every notebook, loads this module.  The directive `#show` displays the signature of `RSet`:
- The type `'a RSet.t` is the type of sets whose elements have the type `'a`.  The type is *polymorphic*, i.e. the same module works for sets of numbers, sets of strings, sets of sets, and so on.
- Internally, a set is represented as a sorted list without duplicates.  Since every set has exactly one such representation, the operators `=` and `<>` as well as the function `compare` work correctly for sets.  As the type `'a RSet.t` is abstract, it is impossible to create a list that is not sorted and pretend that it is a set.
- The toplevel prints sets in the usual mathematical notation.

In [ ]:
#show RSet

In [ ]:
RSet.of_list [1; 2; 3] = RSet.add 1 (RSet.of_list [2; 3])

Let us create a set that contains two sets which are both equal to $\{1\}$.  The resulting set contains only one element:

In [ ]:
let recursive_child_a = RSet.of_list [1]
let recursive_child_b = RSet.add 1 RSet.empty
let recursive_main_set = RSet.of_list [recursive_child_a; recursive_child_b];;
Printf.printf "Size of the set: %d\n" (RSet.cardinal recursive_main_set);
recursive_main_set

## Representing Mathematical Tuples

In addition to sets, theoretical models frequently rely on **tuples**. A tuple is a finite, ordered sequence of elements. Unlike sets, the order of elements in a tuple is strictly preserved, and duplicate elements are allowed.

The most common example of a tuple, which you already know from high school mathematics, is a **coordinate pair** $(x, y)$ in a 2D Cartesian coordinate system.

For two points $P$ and $Q$:
$$ P = (2, 5) $$
$$ Q = (5, 2) $$

Because the order matters, $P \neq Q$. The $x$-coordinate and $y$-coordinate have strictly defined positions.

In TypeScript, the library `recursive-set` needs a special class `Tuple` to represent tuples.  In OCaml, tuples are part of the language.  They are compared structurally and they are *immutable*, i.e. they can not be changed once they have been created.

In [ ]:
(2, 5) = (2, 5)

In [ ]:
(2, 5) = (5, 2)

Let us define a set of coordinates that a robot has already visited, and then check whether a specific coordinate is in that set.

In [ ]:
let visited_coordinates = RSet.of_list [(2, 5)];;
let current_position = (2, 5);;
Printf.printf "Has the robot visited (2, 5) before? -> %b\n" (RSet.mem current_position visited_coordinates)

### Structural Equality versus Physical Equality

In TypeScript, the expression `[2, 5] == [2, 5]` is `false`, since the operator `==` checks whether two arrays are stored at the same address in memory.  This is called *reference equality*.  OCaml has two equality operators:
- `=` checks whether two values are *structurally equal*, i.e. whether they have the same content.
- `==` checks whether two values are *physically equal*, i.e. whether they are stored at the same address in memory.

The two lists below are created separately.  Therefore, they are structurally equal, but not physically equal.  In this lecture, we always use the operator `=`.

In [ ]:
[2; 5] = [2; 5]

In [ ]:
[2; 5] == [2; 5]

### Mutable Data in Sets

In contrast to tuples and lists, *arrays* are mutable in OCaml.  If we store an array in a set and change the array afterwards, the set changes as well.  As the set is still sorted according to the *old* value of the array, it might even violate the invariant of `RSet`.  Therefore, arrays should never be stored in sets.

In [ ]:
let start_point = [| 2; 5 |]
let visited = RSet.of_list [start_point];;
start_point.(0) <- 99;
visited

## Associating Data: Maps

Now that we know how to represent coordinates as tuples, we often want to associate data with these coordinates.  A data structure that associates a *key* with a *value* is called a **map** (or dictionary).

Think of a chess board or a grid: You have a coordinate (the key) and a piece on that coordinate (the value). Mathematically, this is a mapping:
$$ f: (x, y) \rightarrow \text{Value} $$
$$ f((2, 5)) = \text{"Queen"} $$

In TypeScript, the native `Map` compares keys by reference.  Hence the library `recursive-set` provides a class `RecursiveMap`.  In OCaml, we can use the functor `Map.Make`.  Its argument is a module that defines the type `t` of the keys and the function `compare`.  Here we use the polymorphic function `compare`, which compares pairs structurally.  The toplevel answers with the signature of the new module.

In [ ]:
module PosMap = Map.Make (struct type t = int * int let compare = compare end)

In [ ]:
let safe_grid = PosMap.singleton (2, 5) "Queen";;
Printf.printf "Piece at (2, 5) -> %s\n" (PosMap.find (2, 5) safe_grid)

The lookup with the tuple `(2, 5)` succeeds, because the keys are compared structurally.  The keys of such a map can also be sets of type `'a RSet.t`, since `compare` works correctly for these sets.  Keys of a type created by `Set.Make` would only work if the map were created with the function `compare` of the corresponding module.

**Summary:**
1. **Uniqueness:** Use `RSet` when you need a mathematical set without duplicates, in particular for sets of sets.
2. **Sequences:** Use tuples when you need an ordered, immutable pair or sequence (e.g., coordinates, transition pairs).
3. **Mappings:** Use `Map.Make` when you need to assign a value to a key.

# Advanced Operations with `RSet`

Now that we have established how to construct mathematically sound sets and tuples, we can perform actual computations with them. In theoretical computer science, we rarely just store elements; we manipulate them using set theory and functional programming.

## Basic Operations: Add and Remove

The function `RSet.add x s` returns the set $s \cup \{x\}$, and `RSet.remove x s` returns the set $s \setminus \{x\}$.  In contrast to the TypeScript library, these functions do not change the set `s`: All sets of type `'a RSet.t` are *immutable*.

In [ ]:
let alphabet = RSet.of_list ["a"; "b"; "c"];;
print_endline ("Initial alphabet: " ^ RSet.to_string Fun.id alphabet);
let alphabet2 = RSet.remove "b" alphabet in
print_endline ("After removing 'b': " ^ RSet.to_string Fun.id alphabet2);
let alphabet3 = RSet.add "d" alphabet2 in
print_endline ("After adding 'd': " ^ RSet.to_string Fun.id alphabet3)

The function `RSet.to_string f s` converts the set `s` into a string.  The function `f` converts the elements of `s` into strings.  Above, we have used the *identity function* `Fun.id`, which returns its argument unchanged, since the elements are already strings.

The original set `alphabet` has not been changed:

In [ ]:
alphabet

## Immutability

In mathematics, a set like $A = \{1, 2\}$ is a constant concept.  If you use the set $A$ as an element of another set $B$, $A$ **must never change** again.  If you were allowed to add a $3$ to $A$ later, it would silently corrupt the structure of $B$.

The TypeScript library `recursive-set` prevents this with a *freeze contract*: A set is frozen as soon as it is inserted into another set, and it has to be *cloned* before it can be changed.  In OCaml, we do not need such a contract, since the sets of type `'a RSet.t` can never be changed.  Every operation that "modifies" a set returns a new set.

In [ ]:
let state_a  = RSet.of_list [1; 2]
let powerset = RSet.of_list [state_a]
let state_a_modified = RSet.add 3 state_a;;
Printf.printf "Original state_a: %s\n" (RSet.to_string string_of_int state_a);
Printf.printf "New state_a_modified: %s\n" (RSet.to_string string_of_int state_a_modified);
powerset

## Mathematical Set Operations

The module `RSet` provides all standard mathematical set operations.  These operations always return a **new** set, leaving the original sets unmodified.

- **Union** ($A \cup B$): Elements in $A$ or $B$.
- **Intersection** ($A \cap B$): Elements in both $A$ and $B$.
- **Difference** ($A \setminus B$): Elements in $A$ but not in $B$.
- **Symmetric Difference** ($A \bigtriangleup B$): Elements in either $A$ or $B$, but not in both.

In [ ]:
let set_x = RSet.of_list [1; 2; 3]
let set_y = RSet.of_list [3; 4; 5];;
let show = RSet.to_string string_of_int in
Printf.printf "X = %s, Y = %s\n" (show set_x) (show set_y);
Printf.printf "X ∪ Y (Union): %s\n" (show (RSet.union set_x set_y));
Printf.printf "X ∩ Y (Intersection): %s\n" (show (RSet.inter set_x set_y));
Printf.printf "X \\ Y (Difference): %s\n" (show (RSet.diff set_x set_y));
Printf.printf "X △ Y (Symmetric Difference): %s\n" (show (RSet.sym_diff set_x set_y))

### Subset and Superset Relations ($\subseteq$, $\supseteq$)
Two further fundamental relations in set theory are:
- $A \subseteq B$ (*A is a subset of B*): every element of $A$ is also contained in $B$.
- $A \supseteq B$ (*A is a superset of B*): $A$ contains all elements of $B$.

Note that $A = B$ implies both $A \subseteq B$ and $A \supseteq B$ simultaneously.  The call `RSet.subset a b` checks whether $a \subseteq b$.  As $a \supseteq b$ holds iff $b \subseteq a$, there is no separate function for supersets.

In [ ]:
let small = RSet.of_list [2; 3]
let large = RSet.of_list [1; 2; 3; 4; 5];;
let show = RSet.to_string string_of_int in
Printf.printf "%s ⊆ %s: %b\n" (show small) (show large) (RSet.subset small large);
Printf.printf "%s ⊇ %s: %b\n" (show large) (show small) (RSet.subset small large);
Printf.printf "%s ⊆ %s: %b\n" (show large) (show small) (RSet.subset large small)

### Selecting a Random Element
In mathematical proofs, we frequently state: *"Let $x$ be a random element from the set $A$."*
The function `RSet.choose` returns the smallest element of a set.  Hence, it always returns the same element.  To select an arbitrary element, we use the function `RSet.pick_random`.  It uses the random number generator of the module `Random`.  Unless `Random.self_init ()` is called, this generator produces the same sequence of numbers every time the kernel is started.

In [ ]:
set_x

In [ ]:
Printf.printf "X = %s\n" (RSet.to_string string_of_int set_x);
for _ = 1 to 5 do
  Printf.printf "Random element x ∈ X: %d\n" (RSet.pick_random set_x)
done

## Cartesian Product and Powerset

In addition to basic operations, set theory defines operations that generate entirely new, structurally distinct sets. You may recall these concepts from high school combinatorics or probability theory.

### Cartesian Product ($A \times B$)
The Cartesian product combines two sets into a set of all possible ordered pairs, where the first element belongs to $A$ and the second belongs to $B$.

A classic everyday example is finding all possible combinations of two independent choices, such as pairing a main dish with a drink from a menu.

In [ ]:
let main_dishes = RSet.of_list ["Burger"; "Pasta"]
let drinks      = RSet.of_list ["Water"; "Cola"];;
RSet.cartesian_product main_dishes drinks

### Powerset ($\mathcal{P}(A)$ or $2^A$)
The powerset is the set of all possible subsets of $A$, including the empty set ($\emptyset$) and the set $A$ itself. The size of the powerset is always $2^{|A|}$.

A common real-world analogy is choosing optional toppings for a pizza. Given a set of available toppings, you can choose none (the empty set), exactly one, any combination, or all of them. The powerset represents every possible customized pizza you could order.

In [ ]:
let optional_toppings   = RSet.of_list ["Cheese"; "Sauce"; "Olives"]
let all_possible_orders = RSet.powerset optional_toppings

In [ ]:
RSet.cardinal all_possible_orders

*Warning:* The powerset grows exponentially. A set with just 20 elements has a powerset with $1,048,576$ subsets.  The function `RSet.powerset` intentionally raises an exception if you attempt to compute the powerset of a set with more than 20 elements to prevent your computer from running out of memory.

## Mathematical View of Functional Operations

Before using functional operations in OCaml, it is helpful to describe them mathematically.

Let $A$ be a set, let $f : A \to B$ be a function, and let $P(x)$ be a predicate.
A predicate is a statement about an element $x$ that is either true or false.

In OCaml, an expression such as `fun word -> String.length word = 3`
defines a predicate $P(\textrm{word})$.
An expression such as `fun word -> String.uppercase_ascii word`
defines a function $f(\textrm{word})$.

### Set-builder notation
In mathematics, we often describe sets by specifying a property that their elements must satisfy:

$$
\{x \in A \mid P(x)\}
$$

This is read as:
“the set of all elements $x$ in $A$ such that $P(x)$ holds.”

### `RSet.filter p s`
The function `RSet.filter` keeps exactly those elements that satisfy the predicate `p`:

$$
\texttt{RSet.filter}\;P\;A = \{x \in A \mid P(x)\}
$$

In [ ]:
let vocabulary = RSet.of_list ["cat"; "dog"; "bat"; "mouse"; "owl"]
let three_letter_words = RSet.filter (fun word -> String.length word = 3) vocabulary

### `RSet.map f s`
The function `RSet.map` applies a function to every element of a set:

$$
\texttt{RSet.map}\;f\;A = \{f(x) \mid x \in A\}
$$

Note that the result is again a set, so duplicate results are automatically removed.

In [ ]:
String.uppercase_ascii "abc"

In [ ]:
(* We transform every word to uppercase *)
RSet.map String.uppercase_ascii three_letter_words

In [ ]:
(* Example of duplicate reduction during map:
   We map every word to its length.  ("cat" -> 3, "bat" -> 3) *)
RSet.map String.length three_letter_words

### `RSet.for_all p s` and `RSet.exists p s`
These two functions correspond directly to logical quantifiers:

$$
\texttt{RSet.for\_all}\;P\;A \iff \forall x \in A: P(x)
$$

$$
\texttt{RSet.exists}\;P\;A \iff \exists x \in A: P(x)
$$

In [ ]:
let numbers = RSet.of_list [2; 4; 6; 8];;
Printf.printf "Are all numbers even? -> %b\n" (RSet.for_all (fun n -> n mod 2 = 0) numbers);
Printf.printf "Does the set contain an 8? -> %b\n" (RSet.exists (fun n -> n = 8) numbers);
Printf.printf "Does the set contain an odd number? -> %b\n" (RSet.exists (fun n -> n mod 2 <> 0) numbers)

### `RSet.filter_map f s`
This function combines filtering and mapping in a single step.  The function `f` returns a value of type `'b option`: For an element $x$ that is to be dropped, `f x` returns `None`.  Otherwise, it returns `Some y`, where `y` is the transformed element:

$$
\texttt{RSet.filter\_map}\;f\;A = \{y \mid x \in A \wedge f(x) = \texttt{Some}\;y\}
$$

The TypeScript library uses two functions instead, a predicate and a transformation.  In OCaml, a predicate `p` and a transformation `g` are combined into the function `fun x -> if p x then Some (g x) else None`.

In [ ]:
let mixed_numbers = RSet.of_list [1; 2; 3; 4; 5];;
(* Keep only odd numbers AND multiply them by 10 *)
RSet.filter_map (fun n -> if n mod 2 = 1 then Some (n * 10) else None) mixed_numbers

### `RSet.flat_map f s`
If each element $x$ is mapped to an entire set $f(x)$, then `RSet.flat_map` collects all of these sets and merges them into one:

$$
\texttt{RSet.flat\_map}\;f\;A = \bigcup_{x \in A} f(x)
$$

In [ ]:
let natural_numbers = RSet.of_list [1; 2; 3];;
(* For every natural number n, we generate a set containing n and -n. *)
RSet.flat_map (fun n -> RSet.of_list [n; -n]) natural_numbers

If the sets to be merged are given as a list, we can use the function `RSet.unions`, which computes the union of all sets in a list:

In [ ]:
RSet.unions (List.map (fun n -> RSet.of_list [n; -n]) [1; 2; 3])

### `RSet.fold f s init`
The function `RSet.fold` combines all elements into a single value.
Mathematically, this is similar to operations such as summation or product:

$$
\sum_{x \in A} x
\qquad\text{or}\qquad
\prod_{x \in A} x
$$

More generally, `RSet.fold f s init` starts with the value `init` and repeatedly combines the next element `x` of `s` with the value `acc` accumulated so far by computing `f x acc`.  The elements are processed in ascending order.

In [ ]:
let costs = RSet.of_list [10; 20; 30];;
(* We start with an initial value of 0 and add each element to the accumulator *)
Printf.printf "Total sum of costs: %d\n" (RSet.fold (fun current acc -> acc + current) costs 0)